In [ ]:
!pip install huggingface_hub


In [2]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df1_phy = pd.read_csv("hf://datasets/KadamParth/NCERT_Physics_12th/Physics_12th_Cleaned.csv")



# Login using e.g. `huggingface-cli login` to access this dataset
df2_chem = pd.read_csv("hf://datasets/KadamParth/NCERT_Chemistry_12th/Chemsitry_12th_Cleaned.csv")



# Login using e.g. `huggingface-cli login` to access this dataset
df3_bio = pd.read_csv("hf://datasets/KadamParth/NCERT_Biology_12th/Biology_12th_Cleaned.csv")


# Login using e.g. `huggingface-cli login` to access this dataset
df4 = pd.read_csv("hf://datasets/KadamParth/NCERT_Biology_11th/Biology_11th_Cleaned.csv")

df5 = pd.read_csv("hf://datasets/KadamParth/NCERT_Physics_11th/Physics_11th_Cleaned.csv")

df6 = pd.read_csv("hf://datasets/KadamParth/NCERT_Psychology_11th/Psychology_11th_Cleaned.csv")

df7 = pd.read_csv("hf://datasets/KadamParth/NCERT_Psychology_12th/Psychology_12th_Cleaned.csv")

df8 = pd.read_csv("hf://datasets/KadamParth/NCERT_Science_10th/Science_10th_Cleaned.csv")



In [3]:
df1_phy.info() ,df2_chem.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5239 entries, 0 to 5238
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Topic               5239 non-null   object 
 1   Explanation         5239 non-null   object 
 2   Question            5239 non-null   object 
 3   Answer              5239 non-null   object 
 4   Difficulty          5239 non-null   object 
 5   StudentLevel        5239 non-null   object 
 6   QuestionType        5239 non-null   object 
 7   QuestionComplexity  5239 non-null   float64
 8   Prerequisites       5239 non-null   object 
 9   EstimatedTime       5239 non-null   float64
 10  subject             5239 non-null   object 
 11  grade               5239 non-null   int64  
dtypes: float64(2), int64(1), object(9)
memory usage: 491.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4143 entries, 0 to 4142
Data columns (total 12 columns):
 #   Column              Non-Null Co

(None, None)

In [4]:
df_combined = pd.concat([df1_phy, df2_chem,df3_bio,df4,df5,df6,df7,df8], ignore_index=True)

In [5]:
df = df_combined.drop_duplicates().reset_index(drop=True)

In [6]:
df_combined = df_combined.drop_duplicates().reset_index(drop=True)

In [7]:
import re

In [8]:
# Convert text columns to lowercase
text_columns = ["Topic", "Explanation", "Question", "Answer", "Prerequisites", "subject"]
df[text_columns] = df[text_columns].apply(lambda x: x.str.lower())


In [9]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)  # Remove extra spaces
    text = re.sub(r"[^\w\s.,?!]", "", text)  # Keep only words, numbers, and punctuation
    return text.strip()

df[text_columns] = df[text_columns].applymap(clean_text)

C:\Users\aksha\AppData\Local\Temp\ipykernel_25788\2905590953.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[text_columns] = df[text_columns].applymap(clean_text)


In [10]:
# Normalize numerical columns
df["QuestionComplexity"] = df["QuestionComplexity"].clip(0, 10)  # Assume scale 0-10
df["EstimatedTime"] = df["EstimatedTime"].clip(1, 60)  # Assume reasonable limits (1-60 min)

In [11]:
# Format into LLaMA fine-tuning prompt-response pairs
df["Prompt"] = df.apply(
    lambda row: f"Subject: {row['subject']}\nGrade: {row['grade']}\nTopic: {row['Topic']}\n"
                f"Explanation: {row['Explanation']}\nQuestion: {row['Question']}\n",
    axis=1
)

In [12]:
df["Response"] = df["Answer"]

# Save the cleaned dataset
df[["Prompt", "Response"]].to_json("ncert_finetune.json", orient="records", indent=4)

print("Preprocessing complete. Dataset saved as 'ncert_finetune.json'.")

Preprocessing complete. Dataset saved as 'ncert_finetune.json'.


In [13]:
!pip install datasets



Error processing line 1 of C:\Users\aksha\AppData\Local\Programs\Python\Python310\lib\site-packages\vision-1.0.0-py3.10-nspkg.pth:

  Traceback (most recent call last):
    File "C:\Users\aksha\AppData\Local\Programs\Python\Python310\lib\site.py", line 186, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
    File "<frozen importlib._bootstrap>", line 568, in module_from_spec
  AttributeError: 'NoneType' object has no attribute 'loader'

Remainder of file ignored


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="ncert_finetune.json")

In [ ]:
!pip install transformers

In [ ]:
!pip install -U bitsandbytes

In [ ]:

from transformers import LlamaForCausalLM, LlamaTokenizer, AutoTokenizer

model_name = "meta-llama/Llama-2-7b-chat-hf"
# Try using AutoTokenizer for automatic model detection
tokenizer = AutoTokenizer.from_pretrained(model_name)
# If the above doesn't work, try explicitly setting the token
# tokenizer = LlamaTokenizer.from_pretrained(model_name, use_auth_token=True)
model = LlamaForCausalLM.from_pretrained(model_name, device_map="auto", load_in_8bit=True)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

def format_and_tokenize(example):
    prompt = example["Prompt"]
    response = example["Response"]
    text = f"### Instruction:\n{prompt}\n\n### Response:\n{response}"
    return tokenizer(text, truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(format_and_tokenize, batched=False)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # for LLaMA
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="no",
    report_to="none"  # 👈 disables W&B
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
)




In [ ]:
trainer.train()